# ML-07 — Baseline Action Score and Top-20 Review

This notebook builds a transparent rule-based baseline for content-decline prediction,
writes a ranked action queue, and reviews the top picks.

> **Skills loaded:** `building-baselines` + `flyrank/flyrank-data`
> 
> **Lane:** Binary classification — predicting `is_declining` (impressions dropped ≥ 20%)
> 
> **Base rate:** 54.2% declining

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### The rule in plain words

**"A page is worth reviewing if it has meaningful search visibility, it hasn't been
updated recently, and its position is slipping off page 1."**

Specifically:
- **Visible:** impressions_90d ≥ 500 (the page has real search demand)
- **Stale:** days_since_last_update ≥ 180 (not refreshed in 6+ months)
- **Slipping:** avg_position > 10 AND avg_position > 0 (off page 1, with valid data)

The score is: `visible × (stale × log(impressions) + slipping × log(impressions) × 0.5)`

Higher impressions amplify the score (more to lose), staleness weighs more than position
because FlyRank's existing refresh flags use staleness as the primary trigger.

### Reason codes

| Reason code | Meaning |
|---|---|
| `stale_and_slipping` | Updated ≥ 180d ago AND position > 10 on a visible page |
| `stale_visible` | Updated ≥ 180d ago on a visible page (position still OK) |
| `position_slipping` | Position > 10 on a visible page (but recently updated) |
| `visible_only` | Visible but neither stale nor slipping |
| `low_visibility` | Below 500 impressions (score = 0) |

### Signal checks

Before encoding the rule, I verify the two signals it leans on.

In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

print(f"Dataset: {len(df):,} rows × {len(df.columns)} columns")
print(f"Base rate: {df['is_declining'].mean()*100:.1f}% declining")
print()

# ── Signal Check 1: Staleness (flag-linked to FlyRank refresh flags) ──
print("SIGNAL CHECK 1: Staleness → Decline")
print("Claim: Pages not updated in ≥ 180 days decline more often.")
print("Link: staleness behind FlyRank's refresh flags.")
print("=" * 60)

stale_table = df.groupby("freshness_tier").agg(
    n=("is_declining", "count"),
    n_declining=("is_declining", "sum"),
    decline_rate=("is_declining", "mean"),
).copy()
stale_table["decline_rate_pct"] = (stale_table["decline_rate"] * 100).round(1)
tier_order = ["0-30", "31-90", "91-180", "181+"]
stale_table = stale_table.reindex([t for t in tier_order if t in stale_table.index])
print(stale_table[["n", "n_declining", "decline_rate_pct"]].to_string())
print()
print("Verdict: MIXED")
print("  Decline rate rises from 0-30d (51.1%) to 91-180d (61.1%), supporting")
print("  the rule. But 181+ drops to 47.1% (n=174) — very stale pages may have")
print("  already stabilized. Signal is useful in the 0-180 day range.")

Dataset: 30,000 rows × 45 columns
Base rate: 54.2% declining

SIGNAL CHECK 1: Staleness → Decline
Claim: Pages not updated in ≥ 180 days decline more often.
Link: staleness behind FlyRank's refresh flags.
                    n  n_declining  decline_rate_pct
freshness_tier                                      
0-30            20480        10473              51.1
31-90             175          103              58.9
91-180           9171         5604              61.1
181+              174           82              47.1

Verdict: MIXED
  Decline rate rises from 0-30d (51.1%) to 91-180d (61.1%), supporting
  the rule. But 181+ drops to 47.1% (n=174) — very stale pages may have
  already stabilized. Signal is useful in the 0-180 day range.


In [2]:
# ── Signal Check 2: Position (linked to CTR-fix logic) ──
print("\nSIGNAL CHECK 2: Position → Decline")
print("Claim: Pages with avg_position > 10 (off page 1) decline more.")
print("Link: position behind FlyRank's CTR-fix logic.")
print("=" * 60)

pos_df = df[df["avg_position"] > 0].copy()  # exclude 0 = no data
pos_table = pos_df.groupby("position_tier").agg(
    n=("is_declining", "count"),
    n_declining=("is_declining", "sum"),
    decline_rate=("is_declining", "mean"),
).copy()
pos_table["decline_rate_pct"] = (pos_table["decline_rate"] * 100).round(1)
tier_order = ["top_3", "page_1", "striking", "page_3_5", "deep"]
pos_table = pos_table.reindex([t for t in tier_order if t in pos_table.index])
print(pos_table[["n", "n_declining", "decline_rate_pct"]].to_string())
print(f"\nn (excl. avg_position=0): {len(pos_df):,}")
print()
print("Verdict: MIXED")
print("  Decline rate peaks at striking distance (61.0%, pos 11-20), supporting")
print("  the idea that pages slipping off page 1 are at highest risk. But deep")
print("  pages (>50) have the lowest rate (34.4%) — they've already bottomed out.")
print("  The signal is most useful for the page_1-to-striking range.")


SIGNAL CHECK 2: Position → Decline
Claim: Pages with avg_position > 10 (off page 1) decline more.
Link: position behind FlyRank's CTR-fix logic.
                   n  n_declining  decline_rate_pct
position_tier                                      
top_3           1116          551              49.4
page_1         11814         6730              57.0
striking        7304         4452              61.0
page_3_5        7242         4067              56.2
deep            1319          454              34.4

n (excl. avg_position=0): 28,795

Verdict: MIXED
  Decline rate peaks at striking distance (61.0%, pos 11-20), supporting
  the idea that pages slipping off page 1 are at highest risk. But deep
  pages (>50) have the lowest rate (34.4%) — they've already bottomed out.
  The signal is most useful for the page_1-to-striking range.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
# ── Build the baseline score ──
# Score formula: transparent, no fitted weights
#   visible × (stale × log(impressions) + slipping × log(impressions) × 0.5)

df["stale"] = (df["days_since_last_update"] >= 180).astype(int)
df["visible"] = (df["impressions_90d"] >= 500).astype(int)
df["slipping"] = ((df["avg_position"] > 10) & (df["avg_position"] > 0)).astype(int)

df["score"] = df["visible"] * (
    df["stale"] * np.log1p(df["impressions_90d"])
    + df["slipping"] * np.log1p(df["impressions_90d"]) * 0.5
)

# ── Assign ONE reason code per row ──
def assign_reason(row):
    if row["visible"] == 0:
        return "low_visibility"
    if row["stale"] and row["slipping"]:
        return "stale_and_slipping"
    if row["stale"]:
        return "stale_visible"
    if row["slipping"]:
        return "position_slipping"
    return "visible_only"

df["reason_code"] = df.apply(assign_reason, axis=1)

# ── Assign action label ──
action_map = {
    "stale_and_slipping": "refresh_and_reposition",
    "stale_visible": "refresh",
    "position_slipping": "optimize_position",
    "visible_only": "monitor",
    "low_visibility": "monitor",
}
df["action"] = df["reason_code"].map(action_map)

# ── Rank and write CSV ──
df["rank"] = df["score"].rank(method="first", ascending=False).astype(int)

output_cols = [
    "rank", "content_id", "client_id", "score", "reason_code", "action",
    "impressions_90d", "avg_position", "days_since_last_update",
    "content_age_days", "ctr", "is_declining",
]
out = df[output_cols].sort_values("rank")

output_path = Path("../../work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
out.to_csv(output_path, index=False)

print(f"Wrote ranked queue: {output_path}")
print(f"Total rows: {len(out):,}")
print(f"Rows with score > 0: {(out['score'] > 0).sum():,}")
print()

# ── Reason code distribution ──
print("Reason code distribution:")
print(df["reason_code"].value_counts().to_string())
print()

# ── Precision@K ──
base_rate = df["is_declining"].mean()
print(f"Precision@K (base rate = {base_rate*100:.1f}%):")
print("=" * 40)
for k in [10, 20, 50, 100]:
    top_k = out.head(k)
    p_at_k = top_k["is_declining"].mean()
    print(f"  Precision@{k:>3d}: {p_at_k*100:.1f}%  "
          f"({'beats' if p_at_k > base_rate else 'DOES NOT beat'} base rate)")

Wrote ranked queue: ..\..\work\outputs\baseline_action_score.csv
Total rows: 30,000
Rows with score > 0: 9,165

Reason code distribution:
reason_code
low_visibility        13274
position_slipping      9148
visible_only           7561
stale_and_slipping       14
stale_visible             3

Precision@K (base rate = 54.2%):
  Precision@ 10: 100.0%  (beats base rate)
  Precision@ 20: 80.0%  (beats base rate)
  Precision@ 50: 64.0%  (beats base rate)
  Precision@100: 58.0%  (beats base rate)


In [4]:
# ── Write metrics JSON (committable receipt) ──
metrics = {
    "baseline_type": "rule_based",
    "rule": "visible × (stale × log(impr) + slipping × log(impr) × 0.5)",
    "thresholds": {
        "visible": "impressions_90d >= 500",
        "stale": "days_since_last_update >= 180",
        "slipping": "avg_position > 10 and avg_position > 0"
    },
    "base_rate": round(float(base_rate), 4),
    "total_rows": int(len(out)),
    "rows_scored": int((out["score"] > 0).sum()),
}
for k in [10, 20, 50, 100]:
    metrics[f"precision_at_{k}"] = round(float(out.head(k)["is_declining"].mean()), 4)

metrics_path = Path("../../work/outputs/baseline_metrics.json")
metrics_path.parent.mkdir(parents=True, exist_ok=True)
metrics_path.write_text(json.dumps(metrics, indent=2))
print(f"Wrote metrics: {metrics_path}")
print(json.dumps(metrics, indent=2))

Wrote metrics: ..\..\work\outputs\baseline_metrics.json
{
  "baseline_type": "rule_based",
  "rule": "visible \u00d7 (stale \u00d7 log(impr) + slipping \u00d7 log(impr) \u00d7 0.5)",
  "thresholds": {
    "visible": "impressions_90d >= 500",
    "stale": "days_since_last_update >= 180",
    "slipping": "avg_position > 10 and avg_position > 0"
  },
  "base_rate": 0.5421,
  "total_rows": 30000,
  "rows_scored": 9165,
  "precision_at_10": 1.0,
  "precision_at_20": 0.8,
  "precision_at_50": 0.64,
  "precision_at_100": 0.58
}


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

The assignment requires at least the top 10. I review the top 20 for thoroughness.

In [5]:
# ── Top-10 review: action, why, what would make it wrong ──
top10 = out.head(10).copy()

print("TOP-10 REVIEW")
print("=" * 90)
print()

# Show the data first
display_cols = ["rank", "content_id", "score", "reason_code", "action",
                "impressions_90d", "avg_position", "days_since_last_update",
                "is_declining"]
print(top10[display_cols].to_string(index=False))
print()

# One-line review for each
print("Row-by-row review:")
print("-" * 90)
for _, row in top10.iterrows():
    cid_short = row['content_id'][-8:]
    actual = '✓ declining' if row['is_declining'] == 1 else '✗ NOT declining'
    print(f"\n  Rank {row['rank']:>2d} | ...{cid_short} | {actual}")
    print(f"    Action: {row['action']}")
    print(f"    Why: {row['impressions_90d']:,} impressions, position {row['avg_position']},"
          f" {row['days_since_last_update']}d since update → {row['reason_code']}")
    
    # What would make it wrong
    if row['days_since_last_update'] >= 180:
        wrong = ("If this page's traffic is seasonal and the staleness is normal "
                 "for its content cycle, the refresh would be premature.")
    elif row['avg_position'] > 10:
        wrong = ("If the position is temporarily affected by algorithm changes "
                 "rather than content quality, repositioning effort would be wasted.")
    else:
        wrong = "If external factors (domain migration, redirect changes) caused the signal."
    print(f"    What would make it wrong: {wrong}")

TOP-10 REVIEW

 rank           content_id     score        reason_code                 action  impressions_90d  avg_position  days_since_last_update  is_declining
    1 content_cf56e2e2e282 16.544548 stale_and_slipping refresh_and_reposition            61678          19.7                     194             1
    2 content_7368877ea310 16.489917 stale_and_slipping refresh_and_reposition            59472          24.8                     194             1
    3 content_1bfaa38ff26c 15.232303 stale_and_slipping refresh_and_reposition            25715          22.2                     194             1
    4 content_0a91db491d14 14.243279 stale_and_slipping refresh_and_reposition            13299          10.5                     193             1
    5 content_5feee3994adb 13.445316 stale_and_slipping refresh_and_reposition             7812          39.0                     194             1
    6 content_c2d929d83eaa 13.395741 stale_and_slipping refresh_and_reposition             7558  

In [6]:
# ── Extended top-20 review ──
top20 = out.head(20).copy()

print("\nTOP 11-20 REVIEW")
print("=" * 90)
remaining = top20.tail(10)
print(remaining[display_cols].to_string(index=False))
print()

for _, row in remaining.iterrows():
    cid_short = row['content_id'][-8:]
    actual = '✓ declining' if row['is_declining'] == 1 else '✗ NOT declining'
    print(f"\n  Rank {row['rank']:>2d} | ...{cid_short} | {actual}")
    print(f"    Action: {row['action']}")
    print(f"    Why: {row['impressions_90d']:,} impressions, position {row['avg_position']},"
          f" {row['days_since_last_update']}d since update → {row['reason_code']}")
    if row['is_declining'] == 0:
        print(f"    ⚠ FALSE POSITIVE — the rule scored this high but the page is NOT declining.")
    if row['days_since_last_update'] >= 180:
        wrong = ("If seasonal traffic patterns explain the staleness window, "
                 "the refresh is premature.")
    elif row['avg_position'] > 10:
        wrong = ("If position is fluctuating (not trending) or the page serves "
                 "a niche with naturally lower positions.")
    else:
        wrong = "If external domain-level changes caused the signal, not content quality."
    print(f"    What would make it wrong: {wrong}")

print(f"\n\nSummary: {top20['is_declining'].sum()}/20 top picks are actually declining "
      f"(precision@20 = {top20['is_declining'].mean()*100:.0f}%).")


TOP 11-20 REVIEW
 rank           content_id     score        reason_code                 action  impressions_90d  avg_position  days_since_last_update  is_declining
   11 content_bdbec75c1148 10.774668 stale_and_slipping refresh_and_reposition             1316          21.8                     194             0
   12 content_77d4d5930e5e 10.080330 stale_and_slipping refresh_and_reposition              828          18.6                     194             1
   13 content_6226ee6adc91  9.453928 stale_and_slipping refresh_and_reposition              545          17.8                     183             1
   14 content_074ba6ead17b  9.420594 stale_and_slipping refresh_and_reposition              533          48.0                     183             1
   15 content_e3ff1b093148  7.250636      stale_visible                refresh             1408           7.8                     183             1
   16 content_7f116ae1f6f5  6.861711      stale_visible                refresh              95

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [7]:
# ── Identify weak picks in the top 20 ──
weak = top20[top20["is_declining"] == 0]
print("WEAK PICKS (false positives in top 20):")
print("=" * 70)
if len(weak) == 0:
    print("  None — all top 20 are actually declining. Suspicious: look harder.")
else:
    print(f"  {len(weak)} false positive(s):")
    for _, row in weak.iterrows():
        print(f"    Rank {row['rank']}: ...{row['content_id'][-8:]} — "
              f"{row['impressions_90d']:,} impr, pos {row['avg_position']}, "
              f"{row['days_since_last_update']}d stale")
        print(f"      Why it's weak: The rule flagged it because of "
              f"{row['reason_code']}, but the page isn't actually declining.")
        print(f"      Possible explanation: high-volume pages can be stale/slipping")
        print(f"      without actually losing traffic if demand is strong enough.")
print()

# ── Pattern observation ──
print("Pattern observation:")
top10_clients = top10["client_id"].value_counts()
if top10_clients.max() >= 5:
    dominant_client = top10_clients.idxmax()
    print(f"  ⚠ {top10_clients.max()}/10 top picks come from the same client "
        f"({dominant_client[-6:]}).")
    print(f"  This suggests a client-level event (batch content, shared update cycle)")
    print(f"  rather than individual page-level signals. A client-holdout split in the")
    print(f"  model phase will test whether this generalizes.")
else:
    print(f"  Top-10 spread across {len(top10_clients)} clients — no single-client dominance.")
print()

# ── Leakage check ──
print("LEAKAGE CHECK")
print("=" * 70)
leakage_cols = [
    "trend_direction", "trend_pct",
    "impressions_last_30d", "impressions_prev_30d",
    "clicks_last_30d", "clicks_prev_30d",
    "sessions_last_30d", "sessions_prev_30d",
]
features_used = ["impressions_90d", "avg_position", "days_since_last_update"]

print(f"  Features used in score: {features_used}")
print(f"  Leakage columns:        {leakage_cols}")

overlap = set(features_used) & set(leakage_cols)
if overlap:
    print(f"  ✗ LEAKAGE DETECTED: {overlap}")
else:
    print(f"  ✓ No overlap — no leakage from label-derived or future-window columns.")
print()

# Confirm trend_direction was used ONLY to create the label, not as a feature
print("  trend_direction used only to create is_declining label (not as a feature): ✓")
print("  No 30-day comparison columns in the score formula: ✓")
print("  No product flags (is_declining_label) used as features: ✓")

WEAK PICKS (false positives in top 20):
  4 false positive(s):
    Rank 11: ...c75c1148 — 1,316 impr, pos 21.8, 194d stale
      Why it's weak: The rule flagged it because of stale_and_slipping, but the page isn't actually declining.
      Possible explanation: high-volume pages can be stale/slipping
      without actually losing traffic if demand is strong enough.
    Rank 18: ...67c3c89b — 497,727 impr, pos 22.2, 48d stale
      Why it's weak: The rule flagged it because of position_slipping, but the page isn't actually declining.
      Possible explanation: high-volume pages can be stale/slipping
      without actually losing traffic if demand is strong enough.
    Rank 19: ...2b1f9536 — 443,434 impr, pos 27.9, 104d stale
      Why it's weak: The rule flagged it because of position_slipping, but the page isn't actually declining.
      Possible explanation: high-volume pages can be stale/slipping
      without actually losing traffic if demand is strong enough.
    Rank 20: ...1efd6

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.